# 🐱🐶 Transfer Learning — Gatos vs Cachorros

**Projeto:** Classificação de imagens de gatos e cachorros utilizando Transfer Learning com MobileNetV2.

**Objetivo:** Aplicar a técnica de Transfer Learning reutilizando uma rede neural pré-treinada (MobileNetV2, treinada no ImageNet) para classificar imagens do dataset `cats_vs_dogs` do TensorFlow Datasets.

**Conceitos abordados:**
- Transfer Learning (congelamento de camadas + fine-tuning)
- Data Augmentation
- Pré-processamento de imagens
- Avaliação de modelo com métricas e gráficos


## 1. Instalação e Importações

In [ ]:
# Instalação de dependências (caso necessário no Colab)
!pip install tensorflow tensorflow-datasets matplotlib -q

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow versão: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")


## 2. Carregamento do Dataset

Utilizamos o dataset `cats_vs_dogs` do TensorFlow Datasets, que contém milhares de imagens de gatos e cachorros.
- **80%** para treino
- **20%** para validação


In [ ]:
# Carregar o dataset cats_vs_dogs
(ds_train_raw, ds_val_raw), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

num_classes = ds_info.features['label'].num_classes
class_names = ['Gato', 'Cachorro']

print(f"Classes: {class_names}")
print(f"Amostras de treino: {tf.data.experimental.cardinality(ds_train_raw).numpy()}")
print(f"Amostras de validação: {tf.data.experimental.cardinality(ds_val_raw).numpy()}")


## 3. Visualização de Amostras do Dataset

In [ ]:
# Visualizar algumas imagens do dataset
plt.figure(figsize=(12, 8))
for i, (image, label) in enumerate(ds_train_raw.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()], fontsize=12)
    plt.axis('off')
plt.suptitle('Amostras do Dataset - Gatos vs Cachorros', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Pré-processamento das Imagens

Redimensionamos todas as imagens para **160x160** pixels e normalizamos os valores dos pixels para o intervalo **[-1, 1]**, que é o esperado pela MobileNetV2.


In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    """Redimensiona e normaliza a imagem para [-1, 1]."""
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 127.5 - 1  # Normalização para [-1, 1]
    return image, label

# Aplicar pré-processamento
ds_train = ds_train_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
ds_val = ds_val_raw.map(preprocess, num_parallel_calls=AUTOTUNE)

# Configurar batches e cache para performance
ds_train = ds_train.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val = ds_val.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

print("Pré-processamento concluído!")
print(f"Tamanho das imagens: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")


## 5. Data Augmentation

Aplicamos transformações aleatórias nas imagens de treino para aumentar a diversidade dos dados e reduzir overfitting.


In [ ]:
# Camadas de Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

# Visualizar augmentation
plt.figure(figsize=(10, 10))
for image, _ in ds_train.take(1):
    sample_image = image[0]  # Pegar primeira imagem do batch
    for i in range(9):
        augmented = data_augmentation(tf.expand_dims(sample_image, 0))
        plt.subplot(3, 3, i + 1)
        # Desnormalizar para visualização
        plt.imshow((augmented[0].numpy() + 1) / 2)
        plt.axis('off')
plt.suptitle('Exemplos de Data Augmentation', fontsize=14)
plt.tight_layout()
plt.show()


## 6. Transfer Learning com MobileNetV2

Carregamos a **MobileNetV2** pré-treinada no ImageNet e **congelamos suas camadas** para aproveitar os padrões já aprendidos (bordas, texturas, formas, etc.).

Em seguida, adicionamos nossas próprias camadas de classificação no topo.


In [ ]:
# Carregar MobileNetV2 pré-treinada (sem a camada de classificação)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,           # Remove a camada final de classificação
    weights='imagenet'           # Pesos pré-treinados no ImageNet
)

# Congelar todas as camadas do modelo base
base_model.trainable = False

print(f"Modelo base: MobileNetV2")
print(f"Camadas totais: {len(base_model.layers)}")
print(f"Camadas treináveis: {len([l for l in base_model.layers if l.trainable])}")
print(f"Shape de saída: {base_model.output_shape}")


## 7. Construção do Modelo Completo

In [ ]:
# Construir o modelo completo
model = tf.keras.Sequential([
    # Data Augmentation
    data_augmentation,

    # Modelo base pré-treinado (congelado)
    base_model,

    # Camadas de classificação personalizadas
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Classificação binária
])

# Compilar o modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 8. Treinamento — Fase 1 (Feature Extraction)

Nesta fase, treinamos apenas as camadas de classificação que adicionamos, mantendo o modelo base congelado.


In [ ]:
# Fase 1: Treinar somente as camadas de classificação
EPOCHS_PHASE1 = 10

history_phase1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS_PHASE1,
    verbose=1
)

print(f"\nAcurácia final (treino): {history_phase1.history['accuracy'][-1]:.4f}")
print(f"Acurácia final (validação): {history_phase1.history['val_accuracy'][-1]:.4f}")


## 9. Fine-Tuning

Agora descongelamos as **últimas 30 camadas** do modelo base e retreinamos com uma **taxa de aprendizado menor**, permitindo ajustes finos nos pesos para nosso problema específico.


In [ ]:
# Descongelar as últimas camadas do modelo base para fine-tuning
base_model.trainable = True

# Congelar todas exceto as últimas 30 camadas
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total de camadas no modelo base: {len(base_model.layers)}")
print(f"Camadas treináveis agora: {len([l for l in base_model.layers if l.trainable])}")

# Recompilar com learning rate menor para fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Fase 2: Fine-tuning
EPOCHS_PHASE2 = 10
TOTAL_EPOCHS = EPOCHS_PHASE1 + EPOCHS_PHASE2

history_phase2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history_phase1.epoch[-1] + 1,
    verbose=1
)

print(f"\nAcurácia final (treino): {history_phase2.history['accuracy'][-1]:.4f}")
print(f"Acurácia final (validação): {history_phase2.history['val_accuracy'][-1]:.4f}")


## 10. Visualização dos Resultados

In [ ]:
# Combinar históricos das duas fases
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

epochs_range = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de Acurácia
ax1.plot(epochs_range, acc, 'b-o', label='Treino', markersize=4)
ax1.plot(epochs_range, val_acc, 'r-o', label='Validação', markersize=4)
ax1.axvline(x=EPOCHS_PHASE1, color='gray', linestyle='--', label='Início Fine-Tuning')
ax1.set_title('Acurácia por Época', fontsize=13)
ax1.set_xlabel('Época')
ax1.set_ylabel('Acurácia')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico de Loss
ax2.plot(epochs_range, loss, 'b-o', label='Treino', markersize=4)
ax2.plot(epochs_range, val_loss, 'r-o', label='Validação', markersize=4)
ax2.axvline(x=EPOCHS_PHASE1, color='gray', linestyle='--', label='Início Fine-Tuning')
ax2.set_title('Loss por Época', fontsize=13)
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Resultados do Treinamento — Transfer Learning (MobileNetV2)', fontsize=14)
plt.tight_layout()
plt.show()


## 11. Testando Predições

In [ ]:
# Fazer predições em imagens de validação
plt.figure(figsize=(14, 10))

for images, labels in ds_val.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(12):
        plt.subplot(3, 4, i + 1)
        # Desnormalizar para visualização
        img = (images[i].numpy() + 1) / 2
        plt.imshow(img)

        pred_label = 1 if predictions[i][0] > 0.5 else 0
        confidence = predictions[i][0] if pred_label == 1 else 1 - predictions[i][0]
        true_label = labels[i].numpy()

        color = 'green' if pred_label == true_label else 'red'
        plt.title(
            f"Pred: {class_names[pred_label]} ({confidence:.1%})\n"
            f"Real: {class_names[true_label]}",
            color=color, fontsize=10
        )
        plt.axis('off')

plt.suptitle('Predições do Modelo (verde=correto, vermelho=errado)', fontsize=14)
plt.tight_layout()
plt.show()


## 12. Avaliação Final

In [ ]:
# Avaliação no conjunto de validação
loss_final, acc_final = model.evaluate(ds_val, verbose=1)

print(f"\n{'='*40}")
print(f"  RESULTADO FINAL")
print(f"{'='*40}")
print(f"  Loss: {loss_final:.4f}")
print(f"  Acurácia: {acc_final:.4f} ({acc_final:.1%})")
print(f"{'='*40}")


## 13. Salvando o Modelo

In [ ]:
# Salvar o modelo treinado
model.save('modelo_cats_vs_dogs.keras')
print("Modelo salvo com sucesso: modelo_cats_vs_dogs.keras")

# Para carregar depois:
# modelo_carregado = tf.keras.models.load_model('modelo_cats_vs_dogs.keras')


## Conclusão

Neste projeto aplicamos **Transfer Learning** com sucesso:

1. **Modelo base:** MobileNetV2 pré-treinada no ImageNet
2. **Fase 1 (Feature Extraction):** Treinamos apenas as camadas de classificação
3. **Fase 2 (Fine-Tuning):** Descongelamos as últimas 30 camadas e refinamos os pesos
4. **Data Augmentation:** Aplicamos transformações para aumentar a robustez

O Transfer Learning permite alcançar alta acurácia mesmo com datasets menores, aproveitando os padrões já aprendidos por redes treinadas em milhões de imagens.
